# Import needed packages, Models, and Data

In [ ]:
import pandas as pd
import os
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import torch
from transformers import pipeline

# this basically means "use my GPU for machine learning shit"
device = 0 if torch.cuda.is_available() else -1

# connect to database and init a cursor for querying
SF_USR = os.getenv('SF_USR')
SF_KEY = os.getenv('SF_KEY')
SF_ID  = os.getenv('SF_ID')
SF_WH  = os.getenv('SF_WH')
SF_DB  = os.getenv('SF_DB')
SF_SC  = os.getenv('SF_SC')
SF_RL  = os.getenv('SF_RL')
SF_XCT = snowflake.connector.connect(
    user      = SF_USR
   ,password  = SF_KEY
   ,account   = SF_ID
   ,warehouse = SF_WH
   ,database  = SF_DB
   ,schema    = SF_SC
   ,role      = SF_RL
)
CSR = SF_XCT.cursor()

# sentiment analyzer doo-dad instantiation
PIPL_SENT = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment",
    device=device,
    truncation=True,
    max_length = 512 
)
## this sentiment model has the below mapping that indicates the overall sentiment returned
## for verification, see this link:
##      https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment

# named-entity recognition doo-dad instantiation
PIPL_NER = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    tokenizer="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=device,
    batch_size=256 
)

# capture the data what needs analyzing
sample_size_in_percent = 15
query = f"""
with src as (
select content_id
      ,usa_timestamp
      ,detected_languages
      ,post_text 
from {SF_DB}.main.firehose_processed sample system ({sample_size_in_percent})
where detected_languages = '["en"]'
)

select * 
from src
;
"""
CSR.execute(query)
DATA = CSR.fetch_pandas_all()

Device set to use cuda:0
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


# Apply Sentiment and NER Analysis

In [ ]:
sentiment_output = PIPL_SENT(DATA['POST_TEXT'].tolist())
ner_output       = PIPL_NER(DATA['POST_TEXT'].tolist()) 
# last runtime (15%) = 26m 14.7s 

# Write Results to Snowflake

In [4]:
DATA['SENTIMENT_ANALYSIS'] = sentiment_output
DATA['NER_ANALYSIS']       = ner_output
DATA['SAMPLING_PERCENT']   = sample_size_in_percent

write_pandas(SF_XCT, DATA
            ,table_name='NER_SENTANA_OUTPUT_SAMPLE'
            ,database=SF_DB.replace('"', '').upper()
            ,schema=SF_SC.replace('"', '').upper()
           )

/tmp/ipykernel_33957/3960979939.py:5: UserWarning: Dataframe contains a datetime with timezone column, but 'use_logical_type=None'. This can result in dateimes being incorrectly written to Snowflake. Consider setting 'use_logical_type = True'
  write_pandas(SF_XCT, DATA


(True,
 1,
 58320,
 [('agcbaygyfn/file0.txt',
   'LOADED',
   58320,
   58320,
   1,
   0,
   None,
   None,
   None,
   None)])